<a href="https://colab.research.google.com/github/mrsamgary475-boop/Irene-Gallagher/blob/main/Copy_of_LivePortrait_Colab_Free.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Irene neural avatar test (free Google Colab)

This notebook runs the official LivePortrait human model on a temporary Colab GPU. It uploads a source photo, animates it with a short driving clip, previews the result, and lets you download the MP4.

Colab sessions are temporary and free GPU access is not guaranteed. Do not upload anything you do not want processed by Google Colab.

In [ ]:
import os
import shutil
import torch

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU was assigned. In Colab choose Runtime > Change runtime type > T4 GPU, then rerun this cell.')
print('Free disk (GB):', round(shutil.disk_usage('/content').free / 1024**3, 1))

In [ ]:
!git clone -q --depth 1 https://github.com/KlingTeam/LivePortrait.git /content/LivePortrait
%cd /content/LivePortrait
!pip install -q -r requirements.txt
!pip install -q "huggingface_hub[cli]"

In [ ]:
%cd /content/LivePortrait
!huggingface-cli download KlingTeam/LivePortrait --local-dir pretrained_weights --exclude "*.git*" "README.md" "docs"
print('Pretrained weights are ready.')

In [ ]:
from google.colab import files
import os

print('Choose Irene photo (PNG or JPG).')
uploaded = files.upload()
original_filename = next(iter(uploaded))

# Rename the uploaded file to a simpler name without spaces or special characters
new_filename = 'source.' + original_filename.split('.')[-1]
source_path = os.path.join('/content/LivePortrait/', new_filename) # Corrected path
os.rename(os.path.join('/content/LivePortrait/', original_filename), source_path)

print('Source photo:', source_path)

In [ ]:
%cd /content/LivePortrait
!python inference.py -s "{source_path}" -d assets/examples/driving/d9.mp4 -o animations --flag_crop_driving_video --driving_option expression-friendly

In [ ]:
%cd /content/LivePortrait
!ls -l assets/examples/driving/

In [ ]:
from glob import glob
from IPython.display import Video, display
import os # Added explicit import for os

# Specifically look for the animation generated with d9.mp4
output_path = '/content/LivePortrait/animations/source--d9_concat.mp4'

print(f'Checking for animation file: {output_path}')

if not os.path.exists(output_path):
    # If the specific file is not found, try the general glob approach as a fallback
    outputs = sorted(glob('/content/LivePortrait/animations/*.mp4'))

    print(f"Glob found: {outputs}") # Explicitly print the result of glob

    if not outputs:
        # Added a diagnostic print to see what glob finds
        print('No MP4 files found in /content/LivePortrait/animations/. Current directory:', os.getcwd())
        # Ensure the directory exists before listing
        if os.path.exists('/content/LivePortrait/animations'):
            print('Contents of animations directory:', os.listdir('/content/LivePortrait/animations'))
        else:
            print('Directory /content/LivePortrait/animations does not exist.')
        raise FileNotFoundError('LivePortrait did not produce an MP4 file.')
    output_path = outputs[-1]
    print('Result (from glob):', output_path)
else:
    print('Result:', output_path)

display(Video(output_path, embed=True, width=512))

In [ ]:
from google.colab import files
files.download(output_path)

## LivePortrait Resources

Here are some useful links related to LivePortrait:

*   **LivePortrait GitHub Repository:** [https://github.com/KlingTeam/LivePortrait](https://github.com/KlingTeam/LivePortrait)
*   **LivePortrait Project Page:** [https://liveportrait.github.io/](https://liveportrait.github.io/)
*   **HuggingFace Model Page (for pretrained weights):** [https://huggingface.co/KlingTeam/LivePortrait](https://huggingface.co/KlingTeam/LivePortrait)

In [ ]:
import os
import shutil
import torch
from glob import glob
from IPython.display import Video, display
from google.colab import files

In [ ]:
import os
import shutil
import torch
from glob import glob
from IPython.display import Video, display
from google.colab import files

# Install pyngrok for more reliable ngrok tunneling
!pip install pyngrok

from flask import Flask, request, jsonify
from pyngrok import ngrok, conf
import threading
import time
from google.colab import userdata # Import userdata to access secrets

app = Flask(__name__)

# Set ngrok authtoken from Colab secrets
# Make sure you have added your NGROK_AUTH_TOKEN to Colab secrets (click the '🔑' icon in the left panel)
if 'NGROK_AUTH_TOKEN' in userdata.get_secret_keys():
    conf.get_default().auth_token = userdata.get('NGROK_AUTH_TOKEN')
else:
    print("Warning: NGROK_AUTH_TOKEN not found in Colab secrets. ngrok may not start.")

# Function to start ngrok tunnel in a separate thread
def start_ngrok_tunnel():
    # Kill any existing ngrok tunnels
    ngrok.kill()
    try:
        # Connect to ngrok and get the public URL
        public_url = ngrok.connect(5000)
        print(f" * ngrok tunnel available at: {public_url}")
    except Exception as e:
        print(f"Error starting ngrok tunnel: {e}")

@app.route("/irene-brain", methods=["POST"])
def irene_brain():
    data = request.json
    user_message = data.get("message", "")
    # Replace this with your actual logic
    response = {"reply": f"Irene heard: {user_message}"}
    return jsonify(response)

# Start ngrok tunnel in a separate thread before running Flask app
print("Starting ngrok tunnel...")
ngrok_thread = threading.Thread(target=start_ngrok_tunnel)
ngrok_thread.daemon = True
ngrok_thread.start()

# Give ngrok a moment to start
time.sleep(5)

# Run the Flask app
app.run()

In [ ]:
import requests
from pyngrok import ngrok
import json

# Get the public URL of the ngrok tunnel
tunnels = ngrok.get_tunnels()
public_url = None
for tunnel in tunnels:
    if tunnel.proto == 'https': # Or 'http' depending on your tunnel setup
        public_url = tunnel.public_url
        break

if public_url:
    print(f"Ngrok Public URL: {public_url}")
else:
    print("Could not find an active ngrok tunnel. Make sure the Flask app is running.")
    public_url = "YOUR_NGROK_URL_HERE" # Placeholder if not found, user can manually paste

Now, let's send a sample POST request to the `/irene-brain` endpoint using the `public_url` obtained above. You can modify the `payload` to send different messages.

In [ ]:
if 'public_url' in locals() and public_url != "YOUR_NGROK_URL_HERE":
    endpoint_url = f"{public_url}/irene-brain"
    headers = {'Content-Type': 'application/json'}
    payload = {'message': 'Hello Irene, how are you today?'}

    try:
        response = requests.post(endpoint_url, headers=headers, data=json.dumps(payload))
        response.raise_for_status() # Raise an exception for HTTP errors
        print("Response from Irene brain:")
        print(response.json())
    except requests.exceptions.RequestException as e:
        print(f"Error making request: {e}")
else:
    print("Ngrok public URL is not available. Please run cell `ae8a75f` to start the Flask app, then cell `75610e8a` to retrieve the URL.")